# Job Data Cleaning


## Load Data

The loader prefers `MCF-*.json` files anywhere under `../../data`, which keeps the
merged notebook aligned with the project dataset and avoids mixing in unrelated JSON
files when the folder layout changes.


In [1]:
import json
import re
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup


DATA_ROOT = Path("../../data")
OUTPUT_DIR = DATA_ROOT / "cleaned_data"
RAW_SKILLS_OUTPUT = DATA_ROOT / "job_skills_raw.xlsx"
CLEANED_SKILLS_OUTPUT = DATA_ROOT / "job_skills_cleaned_frequency.xlsx"


def clean_html(text):
    if not isinstance(text, str):
        return ""

    cleaned = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    cleaned = cleaned.replace("\xa0", " ")
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def find_job_files(data_root):
    mcf_files = sorted(
        path for path in data_root.rglob("*.json")
        if path.name.startswith("MCF-")
    )
    if mcf_files:
        return mcf_files

    job_dir = data_root / "job"
    if job_dir.exists():
        return sorted(job_dir.glob("*.json"))

    return []


def extract_named_values(items, *keys):
    values = []

    if not isinstance(items, list):
        return values

    for item in items:
        if isinstance(item, dict):
            for key in keys:
                value = item.get(key)
                if value:
                    values.append(str(value).strip())
                    break
        elif isinstance(item, str) and item.strip():
            values.append(item.strip())

    return values


def load_jobs_dataframe(data_root=DATA_ROOT):
    rows = []
    job_files = find_job_files(data_root)

    for file_path in job_files:
        with open(file_path, "r", encoding="utf-8") as handle:
            job = json.load(handle)

        salary_info = job.get("salary") or {}
        salary_type = (salary_info.get("type") or {}).get("salaryType")
        metadata = job.get("metadata") or {}

        rows.append(
            {
                "uuid": job.get("uuid") or file_path.stem,
                "title": job.get("title"),
                "description": clean_html(job.get("description")),
                "minimum_years_experience": job.get("minimumYearsExperience"),
                "skills": extract_named_values(job.get("skills"), "skill"),
                "employment_types": extract_named_values(
                    job.get("employmentTypes"), "employmentType"
                ),
                "position_levels": extract_named_values(
                    job.get("positionLevels"), "position"
                ),
                "flexible_work_arrangements": extract_named_values(
                    job.get("flexibleWorkArrangements"),
                    "flexibleWorkArrangement",
                    "arrangement",
                    "name",
                ),
                "categories": extract_named_values(
                    job.get("categories"), "category"
                ),
                "ssoc_code": job.get("ssocCode"),
                "ssoc_version": job.get("ssocVersion"),
                "salary_minimum": salary_info.get("minimum"),
                "salary_maximum": salary_info.get("maximum"),
                "salary_type": salary_type,
                "posting_date": metadata.get("newPostingDate")
                or metadata.get("originalPostingDate"),
                "expiry_date": metadata.get("expiryDate"),
                "number_of_vacancies": job.get("numberOfVacancies"),
            }
        )

    return pd.DataFrame(rows), job_files


/Applications/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Applications/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
original_jobs_df, job_files = load_jobs_dataframe()

print(f"Discovered raw job JSON files: {len(job_files):,}")
print(f"Loaded job rows: {len(original_jobs_df):,}")

original_jobs_df.head(2)


Discovered raw job JSON files: 22,718
Loaded job rows: 22,718


,uuid,title,description,minimum_years_experience,skills,employment_types,position_levels,flexible_work_arrangements,categories,ssoc_code,ssoc_version,salary_minimum,salary_maximum,salary_type,posting_date,expiry_date,number_of_vacancies
0,4a68226fc8b1d6f39bcf78451f7bc198,Sales Administrator,Serve as administrator / key operator for all ...,1,"[Sales, Microsoft Office, Microsoft Excel, Tra...",[Full Time],[Executive],[],"[Hospitality, Sales / Retail]",33224,2020v2,3000,3200,Monthly,2026-01-25,2026-02-24,1
1,4726f24811127db6967fd1c07ec7ae77,Assistant Field Engineer(Construction),Our Company: Our company develops ICT(Informat...,1,"[Troubleshooting, Construction, Hardware, Elec...",[Full Time],[Fresh/entry level],[],"[Customer Service, Engineering, Professional S...",21499,2020v2,2500,4500,Monthly,2026-01-27,2026-02-26,1


## Job Cleaning
Job-specific notebook:

- restrict to fresh-graduate friendly roles (`0` or `1` year experience)
- remove internships using both text and employment-type signals
- keep only rows with meaningful descriptions
- deduplicate repeated job postings before feature engineering


In [3]:
jobs_df = original_jobs_df.copy()

jobs_df = jobs_df[jobs_df["minimum_years_experience"].isin([0, 1])].copy()
after_experience_filter = len(jobs_df)

jobs_df = jobs_df.dropna(subset=["title", "description"]).copy()

jobs_df["title_clean"] = jobs_df["title"].str.lower()
jobs_df["description_clean"] = jobs_df["description"].str.lower()

valid_desc = jobs_df["description_clean"].str.split().str.len() >= 10
jobs_df = jobs_df[valid_desc].copy()
after_description_filter = len(jobs_df)

intern_mask = (
    jobs_df["title_clean"].str.contains(
        r"\bintern(?:ship)?s?\b",
        regex=True,
        na=False,
    )
    | jobs_df["description_clean"].str.contains(
        r"\bintern(?:ship)?s?\b",
        regex=True,
        na=False,
    )
    | jobs_df["employment_types"].apply(
        lambda values: isinstance(values, list)
        and any("intern" in str(item).lower() for item in values)
    )
)

postgraduate_title_mask = jobs_df["title_clean"].str.contains(
    r"\b(?:postdoc(?:toral)?|research fellow|senior research fellow|assistant professor|associate professor|professor|lecturer|medical doctor)\b",
    regex=True,
    na=False,
)
postgraduate_requirement_mask = jobs_df["description_clean"].str.contains(
    r"(?:require(?:ment|s|d)?|minimum|preferred|qualification|qualifications|possess|holding|hold|must have|with|degree in).{0,120}\b(?:ph\.?d|doctor(?:al|ate)|master(?:'s)? degree|masters degree|master degree|postgraduate|graduate diploma|graduate certificate)\b",
    regex=True,
    na=False,
)

jobs_df = jobs_df[
    ~(intern_mask | postgraduate_title_mask | postgraduate_requirement_mask)
].copy()
after_undergraduate_filter = len(jobs_df)

jobs_df = jobs_df.drop_duplicates(
    subset=["title", "description"],
    keep="first",
).copy()
after_deduplication = len(jobs_df)

jobs_df["has_flexible_work"] = jobs_df["flexible_work_arrangements"].apply(
    lambda values: bool(values) if isinstance(values, list) else False
)
jobs_df["ssoc_3d"] = jobs_df["ssoc_code"].astype(str).str[:3]

print(f"After experience filter: {after_experience_filter:,}")
print(f"After description filter: {after_description_filter:,}")
print(f"After undergrad-only filter: {after_undergraduate_filter:,}")
print(f"After deduplication: {after_deduplication:,}")

jobs_df.head(2)


After experience filter: 9,477
After description filter: 9,476
After undergrad-only filter: 8,834
After deduplication: 7,115


,uuid,title,description,minimum_years_experience,skills,employment_types,position_levels,flexible_work_arrangements,categories,ssoc_code,...,salary_minimum,salary_maximum,salary_type,posting_date,expiry_date,number_of_vacancies,title_clean,description_clean,has_flexible_work,ssoc_3d
0,4a68226fc8b1d6f39bcf78451f7bc198,Sales Administrator,Serve as administrator / key operator for all ...,1,"[Sales, Microsoft Office, Microsoft Excel, Tra...",[Full Time],[Executive],[],"[Hospitality, Sales / Retail]",33224,...,3000,3200,Monthly,2026-01-25,2026-02-24,1,sales administrator,serve as administrator / key operator for all ...,False,332
1,4726f24811127db6967fd1c07ec7ae77,Assistant Field Engineer(Construction),Our Company: Our company develops ICT(Informat...,1,"[Troubleshooting, Construction, Hardware, Elec...",[Full Time],[Fresh/entry level],[],"[Customer Service, Engineering, Professional S...",21499,...,2500,4500,Monthly,2026-01-27,2026-02-26,1,assistant field engineer(construction),our company: our company develops ict(informat...,False,214


## Employment Type And Salary Features

In [4]:
contract_types = ["Permanent", "Contract", "Temporary", "Freelance"]
work_types = ["Full Time", "Part Time"]


def extract_contract_type(values):
    if isinstance(values, list):
        for item in values:
            if item in contract_types:
                return item
    return None


def extract_work_type(values):
    if isinstance(values, list):
        for item in values:
            if item in work_types:
                return item
    return None


jobs_df["contract_type"] = (
    jobs_df["employment_types"].apply(extract_contract_type).fillna("Unknown")
)
jobs_df["work_type"] = (
    jobs_df["employment_types"].apply(extract_work_type).fillna("Unknown")
)

jobs_df = jobs_df[
    ~(
        (jobs_df["contract_type"] == "Unknown")
        & (jobs_df["work_type"] == "Unknown")
    )
].copy()

jobs_df["salary_minimum"] = pd.to_numeric(
    jobs_df["salary_minimum"],
    errors="coerce",
)
jobs_df["salary_maximum"] = pd.to_numeric(
    jobs_df["salary_maximum"],
    errors="coerce",
)
jobs_df["avg_salary"] = jobs_df[
    ["salary_minimum", "salary_maximum"]
].mean(axis=1)

known_work_type = jobs_df["work_type"] != "Unknown"
ssoc_mode = (
    jobs_df[known_work_type]
    .groupby("ssoc_3d")["work_type"]
    .agg(lambda values: values.mode().iat[0] if not values.mode().empty else None)
    .dropna()
)

salary_medians = jobs_df.groupby("work_type")["avg_salary"].median()
full_time_median = salary_medians.get("Full Time")
part_time_median = salary_medians.get("Part Time")

if pd.notna(full_time_median) and pd.notna(part_time_median):
    salary_threshold = (full_time_median + part_time_median) / 2
else:
    salary_threshold = jobs_df["avg_salary"].median()


def impute_work_type(row):
    if row["work_type"] != "Unknown":
        return row["work_type"]

    ssoc_value = ssoc_mode.get(row["ssoc_3d"])
    if pd.notna(ssoc_value):
        return ssoc_value

    if pd.notna(row["avg_salary"]) and pd.notna(salary_threshold):
        return "Full Time" if row["avg_salary"] >= salary_threshold else "Part Time"

    return "Unknown"


jobs_df["work_type"] = jobs_df.apply(impute_work_type, axis=1)

print("Contract types:")
print(jobs_df["contract_type"].value_counts(dropna=False))
print()
print("Work types:")
print(jobs_df["work_type"].value_counts(dropna=False))


Contract types:
contract_type
Unknown      3136
Permanent    2782
Contract      809
Temporary     332
Freelance      56
Name: count, dtype: int64

Work types:
work_type
Full Time    6458
Part Time     657
Name: count, dtype: int64


## Skill Cleaning

Merged skill pipeline:

- raw skill extraction and export
- normalized skill cleanup
- soft-skill collapsing
- fuzzy within-row deduplication
- frequency filtering and `num_skills` thresholding


In [5]:
def base_clean_skill(skill):
    if not isinstance(skill, str):
        return None

    skill = skill.lower().strip()
    return skill or None


jobs_df["skills_clean"] = jobs_df["skills"].apply(
    lambda values: sorted(
        {
            base_clean_skill(skill)
            for skill in values
            if base_clean_skill(skill)
        }
    )
    if isinstance(values, list)
    else []
)

raw_skill_counts = Counter(
    skill for values in jobs_df["skills_clean"] for skill in values
)
raw_skill_freq_df = (
    pd.DataFrame(
        raw_skill_counts.items(),
        columns=["skill", "count"],
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

raw_skill_freq_df.to_excel(RAW_SKILLS_OUTPUT, index=False)

print(f"Saved raw skill frequencies to: {RAW_SKILLS_OUTPUT.resolve()}")
raw_skill_freq_df.head(10)


Saved raw skill frequencies to: /Users/jonlee/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/Academics/Y4S2/DSA4264/2. Project/2. Group Project Deliverables/DSA4264-Project/data/job_skills_raw.xlsx


,skill,count
0,team player,2857
1,customer service,2114
2,interpersonal skills,2059
3,communication skills,1722
4,microsoft office,1677
5,inventory,1511
6,microsoft excel,1461
7,sales,1232
8,leadership,1231
9,administration,1166


In [6]:
core_soft_skills = [
    "communication",
    "leadership",
    "presentation",
    "interpersonal",
    "customer service",
    "problem solving",
    "analytical",
    "teamwork",
    "independence",
]

exact_keep = {
    "project management",
    "management accounting",
    "financial management",
    "risk management",
    "data management",
}

junk_skills = {
    "team player",
    "able to work independently",
    "physically fit",
}


def normalize_skill(skill):
    if not isinstance(skill, str):
        return None

    skill = skill.lower().strip()
    skill = skill.replace("&", "and")
    skill = re.sub(r"[^\w\s]", "", skill)
    skill = re.sub(r"\s+", " ", skill).strip()

    if not skill or skill in junk_skills:
        return None

    if skill in exact_keep:
        return skill

    for core in core_soft_skills:
        if core in skill:
            return core

    skill = re.sub(r"\bskills\b", "", skill)
    skill = re.sub(r"\s+", " ", skill).strip()
    return skill or None


def is_similar(left, right, threshold=0.8):
    return SequenceMatcher(None, left, right).ratio() > threshold


def dedupe_similar_skills(skill_list):
    unique = []

    for skill in skill_list:
        if not any(is_similar(skill, existing) for existing in unique):
            unique.append(skill)

    return unique


jobs_df["skills_clean"] = jobs_df["skills_clean"].apply(
    lambda values: dedupe_similar_skills(
        sorted(
            {
                normalize_skill(skill)
                for skill in values
                if normalize_skill(skill)
            }
        )
    )
    if isinstance(values, list)
    else []
)

cleaned_skill_counts = Counter(
    skill for values in jobs_df["skills_clean"] for skill in values
)
valid_skills = {
    skill
    for skill, count in cleaned_skill_counts.items()
    if count >= 3
}

jobs_df["skills_clean"] = jobs_df["skills_clean"].apply(
    lambda values: [skill for skill in values if skill in valid_skills]
)
jobs_df["num_skills"] = jobs_df["skills_clean"].apply(len)

jobs_df = jobs_df[jobs_df["num_skills"] >= 3].copy()

cleaned_skill_freq_df = (
    pd.DataFrame(
        Counter(
            skill for values in jobs_df["skills_clean"] for skill in values
        ).items(),
        columns=["skill", "count"],
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

cleaned_skill_freq_df.to_excel(CLEANED_SKILLS_OUTPUT, index=False)

print(jobs_df["num_skills"].describe())
print()
print(f"Saved cleaned skill frequencies to: {CLEANED_SKILLS_OUTPUT.resolve()}")
cleaned_skill_freq_df.head(10)


count    7104.000000
mean       12.758868
std         3.218583
min         3.000000
25%        10.000000
50%        13.000000
75%        15.000000
max        20.000000
Name: num_skills, dtype: float64

Saved cleaned skill frequencies to: /Users/jonlee/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/Academics/Y4S2/DSA4264/2. Project/2. Group Project Deliverables/DSA4264-Project/data/job_skills_cleaned_frequency.xlsx


,skill,count
0,communication,2521
1,customer service,2308
2,interpersonal,2080
3,microsoft office,1677
4,inventory,1511
5,microsoft excel,1461
6,leadership,1283
7,sales,1232
8,administration,1166
9,housekeeping,1079


## Save Cleaned Output

The merged notebook saves the cleaned jobs dataset to `../../data/cleaned_data/`.


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

jobs_df = jobs_df.drop(
    columns=[
        "title_clean",
        "description_clean",
        "flexible_work_arrangements",
        "salary_type",
    ],
    errors="ignore",
)

output_path = OUTPUT_DIR / "jobs_cleaned.pkl"

jobs_df.to_pickle(output_path)

print(f"Saved cleaned output to: {output_path.resolve()}")
print(f"Remaining cleaned rows: {len(jobs_df):,}")

jobs_df.head(2)


Saved cleaned output to: /Users/jonlee/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/Academics/Y4S2/DSA4264/2. Project/2. Group Project Deliverables/DSA4264-Project/data/cleaned_data/jobs_cleaned.pkl
Remaining cleaned rows: 7,104


,uuid,title,description,minimum_years_experience,skills,employment_types,position_levels,categories,ssoc_code,ssoc_version,...,posting_date,expiry_date,number_of_vacancies,has_flexible_work,ssoc_3d,contract_type,work_type,avg_salary,skills_clean,num_skills
0,4a68226fc8b1d6f39bcf78451f7bc198,Sales Administrator,Serve as administrator / key operator for all ...,1,"[Sales, Microsoft Office, Microsoft Excel, Tra...",[Full Time],[Executive],"[Hospitality, Sales / Retail]",33224,2020v2,...,2026-01-25,2026-02-24,1,False,332,Unknown,Full Time,3100.0,"[administration, business development, custome...",13
1,4726f24811127db6967fd1c07ec7ae77,Assistant Field Engineer(Construction),Our Company: Our company develops ICT(Informat...,1,"[Troubleshooting, Construction, Hardware, Elec...",[Full Time],[Fresh/entry level],"[Customer Service, Engineering, Professional S...",21499,2020v2,...,2026-01-27,2026-02-26,1,False,214,Unknown,Full Time,3500.0,"[cctv, civil engineering, commissioning, const...",16
